In [ ]:
import numpy as np
from math import sqrt, pi, log
#import OperacoesUnitarias as OP1
#from OperacoesUnitarias import *

# EXEMPLO DE APLICAÇÃO DO PACOTE DE OPERAÇÕES UNITÁRIAS

Todos os exemplos de aplicação são baseados em exercícios retirados da lista de exercícios da disciplina de Operações Unitárias I do professor Ricardo A. Medronho, da UFRJ.

## Dinâmica da partícula e Elutriadores

**EXEMPLO**:

Foram os seguintes resultados obtidos na elutriação de 25g de um pó industrial com água a 30°C, numa vazão de 37 cm3/min:

![alt text](ElutriadorDinamica.png)

**Determine** a distribuição granulométrica da amostra, sabendo-se que $\rho_s$ = 1,8 g/cm3 e $\phi$ = 1,0.

In [ ]:
from OperacoesUnitarias import conversao, Elutriador, DinamicaParticula
from OperacoesUnitarias.propriedades_materiais import p_agua, mu_agua
from OperacoesUnitarias.Granulometria import distribuicao_cumulativa

In [ ]:
massa_total = 25/1000 #Kg
T = 30 + 273 #K
vazao = conversao.cm3pmin_to_m3ps(37) #m3/s

ps = 1800 #Kg/m³
phi = 1.0
D1 = 0.03
D2 = 0.04
D3 = 0.06
D4 = 0.12

## Propriedades água
p = p_agua(T)
mu = mu_agua(T)

## Fração mássica e distribuicao cumulativa
m1 = 4.62/1000 #Kg
m2 = 6.75/1000 #Kg
m3 = 7.75/1000 #Kg
m4 = 4.42/1000 #Kg
m_restante = massa_total - m1 - m2 - m3 - m4
x = np.array([m1, m2, m3, m4, m_restante]) / massa_total
##

## Velocidade da água em cada elutriador (velocidade terminal da maior particula)
u1 = Elutriador.velocidade_fluido(vazao, D1)
u2 = Elutriador.velocidade_fluido(vazao, D2)
u3 = Elutriador.velocidade_fluido(vazao, D3)
u4 = Elutriador.velocidade_fluido(vazao, D4)

##
diametros_corte = []
k1 = DinamicaParticula.k1(phi)
k2 = DinamicaParticula.k2(phi)
for vt in [u1, u2, u3, u4]:
    cd_rep = DinamicaParticula.Cd_Rep(vt, ps, p,  mu)
    Rep = DinamicaParticula.Rep_cd_rep(cd_rep, k1, k2)
    dc = mu*Rep / (p*vt)
    diametros_corte.append(dc)

print(diametros_corte, x, distribuicao_cumulativa(x), sep='\n')

## Câmaras de poeira

**EXEMPLO:**

Pretende-se tratar uma corrente industrial com 1 $m^3/s$ de ar a 100°C e 1 atm contendo finos de carvão (densidade: 1,95 g/$cm^3$ e esfericidade: 0.8), utilizando-se o seguinte sistema (elutriador e câmara de poeira em série):

![alt text](CamaraPoeira.png)

Sabendo-se que o elutriador captura todas as partículas maiores que 200µm e que a eficiência de separação de uma partícula de 80µm na câmara de poeira é igual a 80%, pede-se:

**a)** O diâmetro do elutriador e sua eficiência total de separação.

**b)** A eficiência total de separação da câmara de poeira.

**c)** A eficiência global do processo.


*Dados:* 
$$ \mu_{ar(100°C~,~1 atm)} = 0.021 cP \quad \text{e} \qquad y = \left(\frac{d}{446}\right)^{0.8},~~~\text{d em µm} $$

In [ ]:
from OperacoesUnitarias.Granulometria import GGS
from OperacoesUnitarias import CamaraPoeira, EficienciaSeparacao
from OperacoesUnitarias.propriedades_materiais import ar

In [ ]:
y = GGS(k = 446, m = 0.8)
mu = 0.021/1000
Q = 1
p = ar.MM * 1 / ((273+100) * 0.082)
ps = 1950
phi = 0.8
dc1 = 200*1e-6
d80 = 80*1e-6

# a)
Et1 = 1-y(dc1)

# b)
y = GGS(k = 200, m = 0.8)
d50 = d80 * sqrt(0.5/0.8)
G2 = CamaraPoeira.eficiencia_granulometrica(y.funcao_inversa, d50)
Et2 = EficienciaSeparacao.eficiencia_total(G2) 

# c)
Et = EficienciaSeparacao.eficiencia_global(Et1,Et2)

Et1, Et2, Et

(0.47355123699719215, 0.6247817424063218, 0.8024668124337392)

## Centrífugas

**EXEMPLO:** 

Pretende-se obter um clarificado por meio de centrifugação de uma suspensão viscosa ($\mu$ = 100 cp; $\rho$ = 800 kg/$m^3$) que contém partículas com densidade de 1500 kg/$m^3$. Para isso será testada em laboratório uma centrífuga tubular de dimensões $D_1$ = 1 cm, $D_2$ = 2 cm e L = 15 cm.

**a)** Calcule o diâmetro das partículas que são separadas com uma eficiência de 50%, sabendo-se
que a centrífuga está operando a 23000 rpm, tratando uma vazão de 0,3 L/hr da suspensão
viscosa.

**b)** Calcule o valor do fator $\Sigma$ desta centrífuga.

**c)** Determine qual deve ser a vazão de operação de uma centrífuga industrial (R1 = 5 cm, R2 = 8 cm e L = 80 cm) ao tratar a mesma suspensão a 10.000 rpm para que se obtenha o mesmo diâmetro de corte obtido na centrífuga de laboratório.

**d)** Calcule as eficiências totais produzidas pelas centrífugas de laboratório e industrial, sabendo-se que a distribuição de tamanhos das partículas na suspensão viscosa é y = $d^2$, com d em micra. 


In [ ]:
from OperacoesUnitarias import conversao, Centrifuga, EficienciaSeparacao

In [ ]:
## Dados centrífuga de laboratório:
mu = 0.1
p = 800
ps = 1500
D1 = 0.01
D2 = 0.02
L = 0.15
g = 9.81

## a) d50:
w = conversao.rpm_to_radps(23000)
Q = 0.3/(1000*3600)
R1 = D1/2
R2 = D2/2
V = pi*(R2**2-R1**2)*L
d50 = sqrt(9*mu*Q/((ps-p)*w**2*V)*log(2*R2**2/(R2**2+R1**2)))


## b) de fator Σ:
vtg50 = (ps-p)*g*d50**2/(18*mu)
S = Q/(2*vtg50)


## c) Vazão Q_ind
R1_ind = 0.05
R2_ind = 0.08
L_ind = 0.8
w_ind = conversao.rpm_to_radps(10000)
V_ind = pi*(R2_ind**2-R1_ind**2)*L_ind
S_ind = Centrifuga.sigma('tubular', R1_ind, R2_ind, L_ind, w_ind)
Q_ind = Q * S_ind/S 


## d)
y = lambda d: d**2
d = lambda y: sqrt(y)*1e-6

# Lab
d100 = d50 * sqrt(2 * log(R2/R1)/log(2*R2**2/(R1**2+R2**2)))#Centrifuga.calcular_d100(R1,R2,w,V,Q,ps,p,mu)
G = Centrifuga.eficiencia_granulometrica(d, d50, R1, R2)
Et = EficienciaSeparacao.eficiencia_total(G)

# Ind
d100_ind = Centrifuga.calcular_d100(R1_ind, R2_ind, w_ind, V_ind, Q_ind, ps, p, mu)
G = Centrifuga.eficiencia_granulometrica(d, d50, R1_ind, R2_ind)
Eti = EficienciaSeparacao.eficiencia_total(G)

d50, S, Q_ind*3600*1000, d100, d100_ind, Et, Eti

(4.955923179345032e-07,
 44.467742721898105,
 20.341887108560734,
 8.511413473227991e-07,
 7.970789801319926e-07,
 0.7189063839564309,
 0.7313829888661882)

## Hidrociclones

### Exemplo 1:

Uma bateria de hidrociclones de Bradley, de 6 cm de diâmetro, é utilizado para tratar 50 m3/h de uma suspensão aquosa, a 1% em volume de CaCO3.

Calcular:

**a)** o número de hidrociclones necessários;

**b)** a eficiência total de separação;

**c)** a concentração volumétrica do underflow.

DADOS: $\Delta P$ = 50 psi ; $\rho_s$ = 2.8 g/$cm^3$; $D_u$ = 0.8cm ; T = 20°C

Distribuição granulométrica do material particulado na alimentação:

d ($\mu m$) 5 10 15 20 30 40

y (%) 12 29 46 60 80 92

In [ ]:
from OperacoesUnitarias.Granulometria import melhor_modelo
from OperacoesUnitarias import Hidrociclone, EficienciaSeparacao
from OperacoesUnitarias.propriedades_materiais import p_agua, mu_agua

In [ ]:
## Dados
T = 20 + 273
p = p_agua(T)
mu = mu_agua(T)
dP = conversao.psi_to_pa(50)
ps = 2800
Cv = 0.01
Qt = 50/3600
Dc = 0.06
Du = 0.008
tipo_hidrociclone = Hidrociclone.BRADLEY
d = np.array([5,10,15,20,30,40])
y = np.array([12,29,46,60,80,92])/100

# Ajustando o modelo
modelo = melhor_modelo(y,d)
print(modelo)

## a)
Q = Hidrociclone.calcular_vazao(tipo_hidrociclone, dP, Dc, p, mu)
n = Qt/Q
print(Q, n, p, mu)

## b)
Re = Hidrociclone.reynolds(Q, Dc, p, mu)
Eu = Hidrociclone.Eu(tipo_hidrociclone, Re, Cv)
Rf = Hidrociclone.razao_de_fluido(tipo_hidrociclone, Du, Dc, Eu)
Stk50Eu = Hidrociclone.Stk50Eu(tipo_hidrociclone, Rf, Cv)
d50 = Hidrociclone.calcular_d50(Stk50Eu, Dc, Q, dP, ps, p, mu)
G = Hidrociclone.eficiencia_granulometrica_reduzida(modelo.inversa, d50)
Et = EficienciaSeparacao.eficiencia_total(G, G_reduzida=True, Rf=Rf)
print(d50, Et)

## c)
Cv_underflow = Hidrociclone.calcular_Cv_underflow(Et, Rf, Cv)
Cv_overflow = Hidrociclone.calcular_Cv_overflow(Et, Q, Cv, Cv_underflow)
print(Cv_underflow, Cv_overflow)

### Exemplo 2:

Uma corrente de água contendo sólidos particulados poluentes, gerado no processo de despolpamento do café, passa por um tratamento primário em um hidrociclone Rietema. Obtém-se no fundo uma corrente concentrada e no topo uma corrente diluída. Sabendo-se que: 
- Diâmetro de corte reduzido = 80 $\mu m$
- Densidade do fluído = 1000 kg/$m^3$
- Concentração volumétrica de sólidos na alimentação = 0,05
- Razão de fluido igual a 20%

Distribuição acumulada do tamanho de partículas na alimentação do hidrociclone:

dp: 50 70 100 150 200 250

y(%): 2 10 40 70 95 100

**Determine** as concentrações volumétricas de sólidos no underflow e no overflow

In [ ]:
import sympy as sp
from OperacoesUnitarias import EficienciaSeparacao, Hidrociclone
from OperacoesUnitarias.Granulometria import melhor_modelo

In [ ]:

## Dados:
d50_ = 80*1e-6
p = 1000
cwv = 0.05
Rf = 0.2

## Distribuição:
d = np.array([
    50, 70, 100, 150, 200, 250
])
y = np.array([
    2, 10, 40, 70, 95, 100
])/100

## Ajuste automático do melhor modelo granulométrico:
modelo = melhor_modelo(y[:-1], d[:-1])
print(modelo)
d = modelo.funcao_inversa

## Cálculos:
G = Hidrociclone.eficiencia_granulometrica_reduzida(d, d50_)
Et = EficienciaSeparacao.eficiencia_total(G, G_reduzida=True, Rf=Rf)

# Underflow
cvu = sp.Symbol('cvu')
cvu = sp.solve(Rf - Et*cwv/cvu*(1-cvu)/(1-cwv), cvu)[0]

# Overflow
Q = sp.Symbol('Q')
Qu = Et*cwv/cvu*Q
Qo = Q - Qu
cvo = (Q*cwv - Qu*cvu)/Qo

Et, cvu, cvo

(0.8169977972985905, 0.176954339847556, 0.0118963905933312)

## Ciclones

### Exemplo 1:

Em um processo de Craqueamento Catalítico Fluido (FCC), utilizado para produzir GLP e nafta a partir de gasóleo pesado da torre de destilação a vácuo, o catalisador é circulado no processo, nas etapas de reação e regeneração do mesmo, e seus finos gerados no processo são separados em dois ciclones em série no topo do vaso regenerador, onde é realizado o descoqueamento (regeneração) do catalisador, a fim de que se restabeleça a atividade do mesmo para a reação. Esta corrente de saída superior da regeneradora contém gases de combustão, inertes e finos do catalisador, sendo a composição dos gases, em base molar, a seguinte:

![alt text](Ciclone.png)
*(Massarani, G. (2002), Fluidodinâmica em Sistemas Particulados)*

Considerar a corrente efluente como gás ideal e as propriedades da mistura como uma ponderação
molar dos componentes desta.

A distribuição dos finos na corrente de alimentação segue uma distribuição, segundo o modelo G.G.S. da forma:

$$ y = \left( \frac{d}{150} \right)^{1.25}$$


O catalisador utilizado tem densidade de 0,85 g/cm³, e a premissa adotada é que o primeiro ciclone separe as partículas com diâmetro de 40 $\mu m$ com 98% de eficiência e o que o segundo ciclone separe a partícula de 20 $\mu m$ com 99% de eficiência. 

**Dimensione** os ciclones segundo a geometria Stairmand High Efficiency e **calcule** a eficiência de separação do conjunto.


In [ ]:
from OperacoesUnitarias.Granulometria import GGS
from OperacoesUnitarias import Ciclone, EficienciaSeparacao
from OperacoesUnitarias.propriedades_materiais import ar

In [ ]:
y = GGS(k = 150, m = 1.25)
ps = 850
d98_1 = 40*1e-6
d99_2 = 20*1e-6
Stk50 = 1.19 * 1e-4
Eu = 324
mu = 0.04/1000
T = 650+273
P = 2.5
R = 0.082
p = ar.MM * P / (T*R)
d50_1 = 5.7*1e-6
d50_2 = 2.01*1e-6
G1 = Ciclone.eficiencia_granulometrica(y.funcao_inversa, d50_1)
G2 = Ciclone.eficiencia_granulometrica(y.funcao_inversa, d50_2)
Gconj = lambda y: G1(y)+(1-G1(y))*G2(y)
Et = EficienciaSeparacao.eficiencia_global(Gconj=Gconj)

Et

0.9939983377964622

### Exemplo 2:

Deseja-se estudar o desempenho de uma bateria constituída por 2 ciclones Lapple em série com respectivamente 63,6 cm e 45 cm de diâmetro no tratamento de 27,7 m3/min de gás contendo 3% em volume de sólido.

− Propriedades do gás: densidade 1,1x10-3g/cm3 e viscosidade 1,7x10-2cP;
− Propriedades das partículas sólidas: densidade 2,5 g/cm3 e distribuição granulométrica dada por:

$$ y = 1 - exp\left[-\left(\frac{d}{17.3}\right)^{1.5}\right]~, \quad d~\text{em}~\mu m $$

Pede-se:

**a)** A eficiência global de coleta do sistema.

**b)** A potência do soprador para o serviço.

In [ ]:
from OperacoesUnitarias import conversao, EficienciaSeparacao, Ciclone
from OperacoesUnitarias.Granulometria import RRB

In [ ]:
## Dados
Dc1 = 0.636
Dc2 = 0.45
Q = 27.7/60 #m3/s
Cv = 0.03
p = 1.1
mu = conversao.cp_to_pas(1.7*10**-2) 
ps = 2500
y = RRB(k = 17.3, m = 1.5)
Stk50 = 6.33*10**-4

## Calculando d50
d50_1 = Ciclone.calcular_d50(Stk50,Dc1,Q,ps,p,mu,Cv=Cv,n=4.65)
d50_2 = Ciclone.calcular_d50(Stk50,Dc2,Q,ps,p,mu,Cv=Cv,n=4.65)
print(d50_1, d50_2)

## G
G1 = Ciclone.eficiencia_granulometrica(y.funcao_inversa,d50_1)
G2 = Ciclone.eficiencia_granulometrica(y.funcao_inversa,d50_2)

## Et
Gconj = lambda x: G1(x) + (1-G1(x))*G2(x)
Et_global = EficienciaSeparacao.eficiencia_global(Gconj=Gconj)
print(Et_global)

## b)
dP1 = Ciclone.deltaP(316, Dc1, Q, p)
dP2 = Ciclone.deltaP(316, Dc2, Q, p)
Pot = Ciclone.potencia(dP1+dP2, Q)
print(Pot)

## Meios Porosos

**EXEMPLO:**

Determine a queda de pressão no reator catalítico de leito fixo sabendo-se que este opera isotermicamente a 700 °C e que a pressão na descarga é de 1 atm.

DADOS:
- vazão mássica do gás (propriedades do $O_{2}$): 2 kg/h
- o catalisador forma um leito de 40 cm de diâmetro e 2,0 m de altura, com porosidade 0,44.
- as partículas de catalisador seguem a distribuição GGS de parâmetros k = 185 $\mu m$ e m = 1,8.
- a esfericidade das partículas é 0.75

In [ ]:
from OperacoesUnitarias.Granulometria import GGS
from OperacoesUnitarias import conversao, MeiosPorosos

In [ ]:
## Dados
T = 700 + 273.15
P = conversao.atm_to_pa(1)
mdot = 2/3600
D = 0.4
L = 2
e = 0.44
y = GGS(k=185, m=1.8)
phi = 0.75

# Viscosidade O2 (Incropera - Tabela A.4)
mu = 380.8e-7

## Cálculos
A = pi*D**2/4
G = mdot/A
k = MeiosPorosos.kappa(phi, y.diametro_sauter, e)
c = MeiosPorosos.C(k, e)
pdP = c/sqrt(k)*G**2*L + mu/k*G*L 
P1 = sqrt(conversao.atm_to_pa(1)**2 + 2*pdP*8.314*T/(32/1000))

G, y.diametro_sauter, k, c, pdP, conversao.pressao.pa_to_atm(P1), conversao.pressao.pa_to_atm(P1-P)

## Sedimentadores

Determinar o diâmetro e a altura de um sedimentador para operar com 20 m3/h de uma suspensão aquosa de barita ($\rho_s$=4,1 g/cm3) a 30 °C. A concentração de sólidos na alimentação é de 103 g/L de suspensão e o lodo final deve ter 346 g/L de suspensão. 

O ensaio de proveta a 30 °C conduziu ao seguinte resultado:

*Tempo de sedimentação (t) versus altura da interface de clarificação (z):*

t (min) 0 2 5 10 14 18 23 26 30 33 40 45

z (cm) 40 37 32,4 24,9 18,8 12,6 8,5 7,4 6,3 5,6 4,8 4,5

In [ ]:
import matplotlib.pyplot as plt
from OperacoesUnitarias import Sedimentador

In [ ]:
## Dados
T = 25 + 273.15
Q = 30/3600
ps = 2200
cwv = 80
cwvu = 250

## Experimento
t = np.array([0, 5, 10, 15, 20, 25, 30, 35, 40, 45])*60
z = np.array([40, 32.8, 25.5, 18.8, 14.2, 11.2, 9.6, 6.6, 5.2, 4.0])/100

## Cálculo de Z_min
cv_cvu = cwv/cwvu
z0 = z[0]
zmin = z0 * cv_cvu

## Parte linear:
t_linear = t[:4]
z_linear = z[:4]
reg_lin = np.poly1d(np.polyfit(t_linear, z_linear, 1))

## Parte exp:
t_exp = t[3:]
z_exp = z[3:]
beta = 0.407829333
alpha = np.log(0.99914552)
reg_exp = lambda t: beta * np.exp(alpha * t)
tmin = np.log(zmin/beta)/alpha 

plt.scatter(t, z, color='grey')
plt.plot(np.linspace(t[0], t[3]), reg_lin(np.linspace(t[0], t[3])))
plt.plot(np.linspace(t[3], t[-1]), reg_exp(np.linspace(t[3], t[-1])))
plt.plot(np.linspace(0,1500), [zmin for i in range(50)])
plt.xlabel('t(s)')
plt.ylabel('z(m)')
plt.show()

tu = 2600
tc = 1400

# Diametro
Amin = Q*tmin/z0
Dmin = sqrt(4*Amin/pi)
f1, f2 = Sedimentador.fatores_correcao_area(Dmin) # Só para ver o valor ao final
Aproj = Sedimentador.correcao_area_minima(Amin, Dmin)
Dproj = sqrt(4*Aproj/pi)

# Altura
HCD = Sedimentador.HCD(tu, tc, Q, Amin, cwv, cwvu)
H = Sedimentador.altura_minima(tu, tc, Q, Amin, cwv, cwvu)


zmin, tmin, Amin, Dmin, f1, f2, Aproj, Dproj, HCD, H